In [1]:
import oqs
import os
import hashlib
import json
import cryptography
import struct
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

liboqs-python faulthandler is disabled


# Hybrid Cryptography
####  Core Idea: Use KEM to securely share a key → Use that key with AES to encrypt data → Use ML-DSA to sign & authenticate everything!

In [2]:
# ─────────────────────────────────────────────────────────────
#                       PHASE 1 — SETUP
# ─────────────────────────────────────────────────────────────
print("=" * 60)
print("            PHASE 1 — SETUP")
print("=" * 60)

# ── Alice generates ML-KEM keypair ────────────────────────────
#Alice is the RECEIVER so she owns the KEM keypair
with oqs.KeyEncapsulation("ML-KEM-768") as alice_kem_setup:
    alice_kem_public = alice_kem_setup.generate_keypair()
    alice_kem_secret = alice_kem_setup.export_secret_key()

print("\n Alice — ML-KEM Keypair Generated")
print(f"    KEM Public Key  : {alice_kem_public.hex()[:40]}...")
print(f"    KEM Secret Key  : {alice_kem_secret.hex()[:40]}...")

# ── Bob generates ML-DSA keypair ──────────────────────────────
# Bob is the SENDER, so he owns the DSA keypair for signing
with oqs.Signature("ML-DSA-65") as bob_dsa_setup:
    bob_dsa_public = bob_dsa_setup.generate_keypair()
    bob_dsa_secret = bob_dsa_setup.export_secret_key()

print("\n Bob — ML-DSA Keypair Generated")
print(f"    DSA Public Key  : {bob_dsa_public.hex()[:40]}...")
print(f"    DSA Secret Key  : {bob_dsa_secret.hex()[:40]}...")

# ── Key Exchange ───────────────────────────────────────────────
print("\n Alice  ─ KEM_pub  ──>  Bob   (so Bob can encrypt TO Alice)")
print(" Bob    ─ DSA_pub  ──>  Alice (so Alice can VERIFY Bob's messages)")
print("\n Setup Complete!\n")

            PHASE 1 — SETUP

 Alice — ML-KEM Keypair Generated
    KEM Public Key  : 153631d2ba7c5bbb3dadf03627292ee88ac74b4c...
    KEM Secret Key  : a08016cba1cd886cbfe6b378d29308f5d04690e8...

 Bob — ML-DSA Keypair Generated
    DSA Public Key  : c061278751594be4d3cbba49e912c98690f49a47...
    DSA Secret Key  : c061278751594be4d3cbba49e912c98690f49a47...

 Alice  ─ KEM_pub  ──>  Bob   (so Bob can encrypt TO Alice)
 Bob    ─ DSA_pub  ──>  Alice (so Alice can VERIFY Bob's messages)

 Setup Complete!



In [3]:
# ─────────────────────────────────────────────────────────────
#                   PHASE 2 — BOB ENCRYPTS AND SIGNS
# ─────────────────────────────────────────────────────────────

print("=" * 60)
print("         PHASE 2 — BOB ENCRYPTS & SIGNS")
print("=" * 60)

message = b"Hello Alice! This is Quantum Safe secret message from Bob!"
print(f"\n   Original Message    : {message.decode()}")

# ── STEP 1: KEM — Encapsulate using Alice's KEM public key ────
with oqs.KeyEncapsulation("ML-KEM-768") as bob_kem:
    kem_ciphertext, shared_secret = bob_kem.encap_secret(alice_kem_public)

print(f"\n  STEP 1 — KEM Encapsulation")
print(f"   Shared Secret      : {shared_secret.hex()[:40]}...")
print(f"   KEM Ciphertext     : {kem_ciphertext.hex()[:40]}...")

# ── STEP 2: AES-256-GCM — Encrypt message with shared secret ──
aes_key = shared_secret[:32]   # 32bytes = AES256
nonce = os.urandom(12)         # 96bit random nonce
aesgcm = AESGCM(aes_key)
ciphertext = aesgcm.encrypt(nonce, message, None)

print(f"\n  STEP 2 — AES-256-GCM Encryption")
print(f"   AES Key (32B)      : {aes_key.hex()[:40]}...")
print(f"   Nonce (12B)        : {nonce.hex()}")
print(f"   AES Ciphertext     : {ciphertext.hex()[:40]}...")

# ── STEP 3: ML-DSA — Sign (kem_ct + nonce + aes_ct) ──────────
payload_to_sign = kem_ciphertext + nonce + ciphertext

with oqs.Signature("ML-DSA-65", bob_dsa_secret) as bob_signer:
    signature = bob_signer.sign(payload_to_sign)

print(f"\n  STEP 3 — ML-DSA-65 Signing")
print(f"   Payload Signed     : kem_ct + nonce + aes_ct")
print(f"   Signature          : {signature.hex()[:40]}...")

# ── Package Bob sends to Alice ─────────────────────────────────
package = {
    "kem_ciphertext": kem_ciphertext,
    "nonce"         : nonce,
    "ciphertext"    : ciphertext,
    "signature"     : signature
} 

print(f"\n  Package Dispatched to Alice!")
print(f"   Contains           : kem_ct + nonce + aes_ct + signature")
print("\n   Bob's Side Complete!\n")

         PHASE 2 — BOB ENCRYPTS & SIGNS

   Original Message    : Hello Alice! This is Quantum Safe secret message from Bob!

  STEP 1 — KEM Encapsulation
   Shared Secret      : 1c6c1c410f765a931c2f1f7eba759f381e7e49eb...
   KEM Ciphertext     : 488a316094f27a38b0499e629a4ed0aaefad0d3b...

  STEP 2 — AES-256-GCM Encryption
   AES Key (32B)      : 1c6c1c410f765a931c2f1f7eba759f381e7e49eb...
   Nonce (12B)        : b8a3df395ac77ae56ee1cdac
   AES Ciphertext     : edef2cafb3b9ee65853532944065bac9762d2be5...

  STEP 3 — ML-DSA-65 Signing
   Payload Signed     : kem_ct + nonce + aes_ct
   Signature          : 1921c514dfb710f8bf31e205238352d6109b5758...

  Package Dispatched to Alice!
   Contains           : kem_ct + nonce + aes_ct + signature

   Bob's Side Complete!



In [4]:
# ─────────────────────────────────────────────────────────────
#                PHASE 3 — ALICE VERIFIES AND DECRYPTS
# ─────────────────────────────────────────────────────────────

# ── STEP 1: ML-DSA — Verify signature using Bob's DSA public key

payload_to_verify = (
    package["kem_ciphertext"]+
    package["nonce"]+
    package["ciphertext"]
)

with oqs.Signature("ML-DSA-65") as verifier:
    is_valid = verifier.verify(
        payload_to_verify,
        package["signature"],
        bob_dsa_public                      #Bob's DSA public key
    )

print(f"\n  STEP 1 — ML-DSA-65 Signature Verification")
print(f"   Verified with      : Bob's DSA Public Key")
print(f"   Signature Valid    : {is_valid}")

if not is_valid:
    raise Exception("  Signature FAILED! Message has been tampered!")

# ── STEP 2: KEM — Decapsulate using Alice's KEM secret key ────

with oqs.KeyEncapsulation("ML-KEM-768", alice_kem_secret) as alice_recv:
    recovered_secret = alice_recv.decap_secret(package["kem_ciphertext"])
    
print(f"\n  STEP 2 — KEM Decapsulation")
print(f"   Decapsulated with  : Alice's KEM Secret Key")
print(f"   Recovered Secret   : {recovered_secret.hex()[:40]}...")

# ── Verify shared secrets match ────────────────────────────────
secrets_match = (shared_secret == recovered_secret)
print(f"   Secrets Match      : {secrets_match}")

# ── STEP 3: AES-256-GCM — Decrypt the message ─────────────────
aes_key_recvd = recovered_secret[:32]
aesgcm_recvd = AESGCM(aes_key_recvd)
decrypted = aesgcm_recvd.decrypt(
    package["nonce"],
    package["ciphertext"],
    None
)

print(f"\n  STEP 3 — AES-256-GCM Decryption")
print(f"   Decrypted Message  : {decrypted.decode()}")
print("\n  Alice's Side Complete!\n")


  STEP 1 — ML-DSA-65 Signature Verification
   Verified with      : Bob's DSA Public Key
   Signature Valid    : True

  STEP 2 — KEM Decapsulation
   Decapsulated with  : Alice's KEM Secret Key
   Recovered Secret   : 1c6c1c410f765a931c2f1f7eba759f381e7e49eb...
   Secrets Match      : True

  STEP 3 — AES-256-GCM Decryption
   Decrypted Message  : Hello Alice! This is Quantum Safe secret message from Bob!

  Alice's Side Complete!



In [5]:
# ─────────────────────────────────────────────────────────────
#                PHASE 4 — TAMPER DETECTION TEST
# ─────────────────────────────────────────────────────────────

#Attacker modifies the ciphertext

tampered_package = dict(package)
tampered_package["ciphertext"] = b"TAMPERED!!" + package["ciphertext"]

tampered_payload = (
    tampered_package["kem_ciphertext"] +
    tampered_package["nonce"]          +
    tampered_package["ciphertext"]
)

with oqs.Signature("ML-DSA-65") as verifier:
    tampered_valid = verifier.verify(
        tampered_payload,
        tampered_package["signature"],
        bob_dsa_public
    )

print(f"\n  Tampered Signature Valid : {tampered_valid}")

if not tampered_valid:
    print("  Attack Detected! Message rejected by Alice!")
print("\n  Tamper Test Complete!\n")


  Tampered Signature Valid : False
  Attack Detected! Message rejected by Alice!

  Tamper Test Complete!



In [6]:
print("=" * 60)
print("         📊  PHASE 5 — SUMMARY REPORT")
print("=" * 60)

print(f"""
┌─────────────────────────────────────────────────────┐
│                Hybrid Encryption Summary            │
├──────────────────────────┬──────────────────────────┤
│ KEM Algorithm            │ ML-KEM-768               │
│ Signature Algorithm      │ ML-DSA-65                │
│ Symmetric Algorithm      │ AES-256-GCM              │
├──────────────────────────┼──────────────────────────┤
│ Alice owns               │ ML-KEM keypair           │
│ Bob owns                 │ ML-DSA keypair           │
├──────────────────────────┼──────────────────────────┤
│ KEM Ciphertext Size      │ {len(kem_ciphertext)} bytes               │
│ AES Ciphertext Size      │ {len(ciphertext)} bytes                 │
│ Signature Size           │ {len(signature)} bytes               │
│ Nonce Size               │ {len(nonce)} bytes                 │
├──────────────────────────┼──────────────────────────┤
│ Signature Verified       │ {is_valid}                     │
│ Secrets Match            │ {secrets_match}                     │
│ Tamper Detected          │ {not tampered_valid}                     │
│ Decryption Successful    │ {decrypted == message}                     │
└──────────────────────────┴──────────────────────────┘
""")


         📊  PHASE 5 — SUMMARY REPORT

┌─────────────────────────────────────────────────────┐
│                Hybrid Encryption Summary            │
├──────────────────────────┬──────────────────────────┤
│ KEM Algorithm            │ ML-KEM-768               │
│ Signature Algorithm      │ ML-DSA-65                │
│ Symmetric Algorithm      │ AES-256-GCM              │
├──────────────────────────┼──────────────────────────┤
│ Alice owns               │ ML-KEM keypair           │
│ Bob owns                 │ ML-DSA keypair           │
├──────────────────────────┼──────────────────────────┤
│ KEM Ciphertext Size      │ 1088 bytes               │
│ AES Ciphertext Size      │ 74 bytes                 │
│ Signature Size           │ 3309 bytes               │
│ Nonce Size               │ 12 bytes                 │
├──────────────────────────┼──────────────────────────┤
│ Signature Verified       │ True                     │
│ Secrets Match            │ True                     │
│ Tamper D